In [ ]:
import jax
import jax.numpy as jnp
from jax import grad, vmap, jit, random, debug

import diffrax

import numpy as np

import skewerjax as sjx


import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
plt.style.use('tableau-colorblind10')
font, rcnew = sjx.config.setup_matplotlib.matplotlib_default_config()
plt.rc('font', **font)
plt.rcParams.update(rcnew)
%matplotlib widget

In [ ]:
Gpc_to_m = 3.085678e25  # 1 Gpc in meters
Gyr_to_s = 3.1536e16  # 1 Gyr in seconds

H0 = 70.0 * Gyr_to_s * Gpc_to_m**-1 * 1e6  # Convert from km/s/Mpc to 1/Gyr
cc = 299_792_458 * Gyr_to_s * Gpc_to_m**-1 # Convert from km/s to Gpc/Gyr

In [ ]:
def ODEs(xx, aa, params):
    H0, omega_r, omega_m, omega_k, omega_lambda = params
    kk = -omega_k/cc**2
    da_dx = (H0/cc) * jnp.sqrt(omega_r + omega_m*aa**1 + omega_k*aa**2 + omega_lambda*aa**4) / jnp.sqrt(1 - kk*xx**2)
    return da_dx


def solve_ODES(params, ICs, x_eval, dx0=None, solver=diffrax.Tsit5(), rtol=1e-4, atol=1e-4, max_steps=1000):

    term = diffrax.ODETerm(ODEs)
    solution_obj = diffrax.diffeqsolve(
        term,
        solver=solver,
        t0=x_eval[0],
        t1=x_eval[-1],
        dt0=dx0,
        y0=ICs,
        args=params,
        saveat=diffrax.SaveAt(ts=x_eval),
        stepsize_controller=diffrax.PIDController(rtol=rtol, atol=atol),
        max_steps=max_steps,
    )
    return solution_obj.ys

# Order of the entries: H0, omega_r, omega_m, omega_k, omega_lambda, kk
cosmologies = jnp.array([
    (H0, 1.0, 0.0, 0.0, 0.0),
    (H0, 0.0, 1.0, 0.0, 0.0),
    (H0, 0.0, 0.0, 0.0, 1.0),
    (H0, 0.0, 0.0, 1.0, 0.0),
    # (H0, 0.0, 0.0, -1.0, 0.0)
])

a0 = 1.0
x0 = 0.0

# initial conditions
ICs = jnp.array([
    [a0],
    [a0],
    [a0],
    [a0],
    # [a0]
])

NN_x = 1000

xf = x0 + (cc) / (a0 * H0) # Dark Energy Domination divergence
# xf = 0.27
xf -= 0.01

xi = x0 - (2 * cc / H0) * (a0)**0.5 # Dark Matter Fomination a' = 0
# xi = -0.27
xi += 0.01

x_forward = jnp.linspace(x0, xf, NN_x)
results_forward = vmap(solve_ODES, in_axes=(0, 0, None))(
    cosmologies, ICs, x_forward
)

x_backward = jnp.linspace(x0, xi, NN_x)
results_backward = vmap(solve_ODES, in_axes=(0, 0, None))(
    cosmologies, ICs, x_backward
)

In [ ]:
x_analytic = jnp.linspace(xi, xf, 10000)

omega_k_positive = 1.0
omega_k_negative = -1.0
analytic_solutions = {
    "Radiation": (x_analytic, a0 + (H0/cc)*(x_analytic - x0)),
    "Matter": (x_analytic, (a0**0.5 + (H0/(2*cc))*(x_analytic - x0))**2),
    "Lambda": (x_analytic, (a0**-1 - (H0/cc)*(x_analytic - x0))**-1),
    "Curvature": (x_analytic, jnp.exp( (H0/(cc*jnp.sqrt(jnp.abs(-omega_k_positive/cc**2))))*(jnp.arcsinh(jnp.sqrt(jnp.abs(-omega_k_positive/cc**2))*x_analytic) - jnp.arcsinh(jnp.sqrt(jnp.abs(-omega_k_positive/cc**2))*x0)) +jnp.log(a0) ) ),
    # "Curvature_omega_k_negative": (x_analytic, jnp.exp( (H0/(cc*jnp.sqrt(jnp.abs(-omega_k_negative/cc**2))))*(jnp.arcsin(jnp.sqrt(jnp.abs(-omega_k_negative/cc**2))*x_analytic) - jnp.arcsin(jnp.sqrt(jnp.abs(-omega_k_negative/cc**2))*x0)) +jnp.log(a0) ) )
}

a_forward = np.array(results_forward[..., 0])
a_backward = np.array(results_backward[..., 0])
numerical_solutions = {}
for ii, key_cosmo in enumerate(list(analytic_solutions.keys())):
    numerical_solutions[key_cosmo] = (
        jnp.concatenate([x_backward[::-1], x_forward]),
        jnp.concatenate([a_backward[ii][::-1], a_forward[ii]])
    )


In [ ]:
# Define color map for cosmologies
cosmo_colors = {
    "Radiation": "crimson",
    "Matter": "royalblue",
    "Lambda": "limegreen",
    "Curvature": "orange",
    "Curvature_omega_k_negative": "purple"
}

In [ ]:
# Setup figure for null geodesics
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7), sharex=True, gridspec_kw={'height_ratios': [3, 1]})

# Plot each cosmology
handles_cosmo = []
for ii, key_cosmo in enumerate(analytic_solutions.keys()):

    color = cosmo_colors[key_cosmo]

    x_analytic, a_analytic = analytic_solutions[key_cosmo]
    ax1.plot(jax.device_get(x_analytic), jax.device_get(a_analytic), linestyle='--', color=color, lw=2)

    x_num, a_num = numerical_solutions[key_cosmo]
    ax1.plot(jax.device_get(x_num), jax.device_get(a_num), linestyle='-', color=color, lw=1)

    # Interpolate analytic solution
    a_interp = jnp.interp(x_num, jax.device_get(x_analytic), jax.device_get(a_analytic))
    ratio = a_num / a_interp
    ax2.plot(x_num, ratio-1, color=color, lw=1.2)


    handles_cosmo.append(Line2D([0], [0], color=color, lw=2, linestyle='-', label=key_cosmo))

# Legends
legend1 = ax1.legend(handles=handles_cosmo, loc='upper left', title="Cosmology", title_fontsize=12, fontsize=11, ncol=2)
ax1.add_artist(legend1)

handles_style = [
    Line2D([0], [0], color='black', lw=2, linestyle='-', label='Numerical'),
    Line2D([0], [0], color='black', lw=2, linestyle='--', label='Analytical')
]
legend2 = ax1.legend(handles=handles_style, loc='lower right', title="Method", title_fontsize=12, fontsize=11)

# === Present day ===
ax1.plot(x0, a0, marker='x', color='black', markersize=8, mew=2)
ax1.text(x0 - .2, a0 - 0.1, "Present day", color='black', fontsize=11, va='bottom', ha='left')
ax1.axvline(x0, color='gray', linestyle='--', lw=0.8)
ax1.axhline(a0, color='gray', linestyle='--', lw=0.8)

# Configure main plot
ax1.set_ylabel("Scale factor $a$ [adim]")

distance_to_andromeda = 765.0 * 10e-6 # Gpc
ax1.axhline(-200 * distance_to_andromeda, color='k', linestyle='--', lw=1.5)

# Ratio plot
ax2.axhline(1.0, color='gray', linestyle='--', lw=0.8)
ax2.set_xlabel("Comoving distance $x$")
ax2.set_ylabel("Residual")
# ax2.set_ylim(0.999, 1.001)

# === Divergence in DE domination ===
ax1.axvline(x0 + (cc) / (a0 * H0), color='limegreen', linestyle='--', lw=0.8)
ax2.axvline(x0 + (cc) / (a0 * H0), color='limegreen', linestyle='--', lw=0.8)
ax1.text(x0 + (cc) / (a0 * H0), 0.05, "DE domin: Divergence", 
         rotation=90, va='bottom', ha='right', color='limegreen', fontsize=8)

# # === a' = 0 in DM domination (positive root) ===
# ax1.axvline(x0 + (2 * cc / H0) * (a0)**0.5, color='royalblue', linestyle='--', lw=0.8)
# ax2.axvline(x0 + (2 * cc / H0) * (a0)**0.5, color='royalblue', linestyle='--', lw=0.8)
# ax1.text(x0 + (2 * cc / H0) * (a0)**0.5, 0.05, "a' = 0 (DM, +)", 
#          rotation=90, va='bottom', ha='right', color='royalblue', fontsize=8)

# === a' = 0 in DM domination (negative root) ===
ax1.axvline(x0 - (2 * cc / H0) * (a0)**0.5, color='royalblue', linestyle='--', lw=0.8)
ax2.axvline(x0 - (2 * cc / H0) * (a0)**0.5, color='royalblue', linestyle='--', lw=0.8)
ax1.text(x0 - (2 * cc / H0) * (a0)**0.5, 0.05, "DM domin: " + r"$a' = 0$",
         rotation=90, va='bottom', ha='right', color='royalblue', fontsize=8)

# === ln(a) = 0 in Radiation domination ===
ax1.axvline(x0 - a0*cc/H0, color='red', linestyle='--', lw=0.8)
ax2.axvline(x0 - a0*cc/H0, color='red', linestyle='--', lw=0.8)
ax1.text(x0 - a0*cc/H0, 0.05, "Rad. domin: Divergence",
         rotation=90, va='bottom', ha='right', color='red', fontsize=8)

ax1.set_yscale('log')
ax2.set_yscale('symlog', linthresh=1e-5)  # linthresh sets the range around 0 where it is linear

plt.tight_layout()
plt.show()